In [ ]:
import pandas as pd
import numpy as np
import pickle
import re
import warnings
from pathlib import Path
from sklearn.linear_model import LinearRegression
from IPython.display import display

warnings.simplefilter("ignore", category=pd.errors.PerformanceWarning)

In [ ]:
# load week 8 model and supporting data
projectDir = Path.cwd().resolve().parent

payload = pickle.load(open(projectDir / "models" / "week8_best_model.pkl", "rb"))
bestModel = payload["model"]
featureCols = payload["featureCols"]
modelThreshold = payload["threshold"]

summary26 = pd.read_csv(projectDir / "data" / "external" / "summary26.csv")
offense26 = pd.read_csv(projectDir / "data" / "external" / "offense26.csv")
defense26 = pd.read_csv(projectDir / "data" / "external" / "defense26.csv")
misc26 = pd.read_csv(projectDir / "data" / "external" / "misc26.csv")
spellings = pd.read_csv(projectDir / "data" / "raw" / "MTeamSpellings.csv")
teamSeason = pd.read_csv(projectDir / "data" / "processed" / "teamSeason.csv")
rawGameStats = pd.read_csv(projectDir / "data" / "raw" / "MarchMadnessGameStats2003-2024.csv")

In [ ]:
def normalizeName(name):
    return re.sub(r"[^a-z0-9]+", "", str(name).lower())


manualExternalToTeamId = {
    "connecticut": 1163,
    "ohio st": 1326,
    "michigan st": 1277,
    "north dakota st": 1295,
    "kennesaw st": 1244,
    "liu": 1254,
    "wright st": 1460,
    "iowa st": 1235,
    "tennessee st": 1398,
    "utah st": 1429,
    "miami fl": 1274,
    "miami oh": 1275,
    "northern iowa": 1320,
    "hawaii": 1218,
    "queens": 1474,
    "texas a&m corpus chris": 1394,
    "saint francis": 1384,
    "southeast missouri": 1369,
}

manualInputToTeamId = {
    "Ohio State": 1326,
    "Michigan State": 1277,
    "North Dakota State": 1295,
    "Hawai'i": 1218,
    "Kennesaw State": 1244,
    "Long Island University": 1254,
    "Wright State": 1460,
    "Iowa State": 1235,
    "Tennessee State": 1398,
    "Utah State": 1429,
    "Miami (Ohio)": 1275,
    "UNI": 1320,
    "UConn": 1163,
    "Miami (Fla.)": 1274,
    "Queens": 1474,
}

spellMap = {
    normalizeName(name): teamId
    for name, teamId in spellings[["TeamNameSpelling", "TeamID"]].drop_duplicates().itertuples(index=False)
}


def resolveTeamId(name):
    return manualInputToTeamId.get(name, spellMap.get(normalizeName(name)))


season2026 = summary26.merge(offense26, on=["Season", "TeamName"], how="left", suffixes=("", "_off"))
season2026 = season2026.merge(defense26, on=["Season", "TeamName"], how="left", suffixes=("", "_def"))
season2026 = season2026.merge(misc26, on=["Season", "TeamName"], how="left", suffixes=("", "_misc"))
season2026["TeamID"] = season2026["TeamName"].map(lambda x: spellMap.get(normalizeName(x)))

for extName, teamId in manualExternalToTeamId.items():
    season2026.loc[season2026["TeamName"].map(normalizeName) == normalizeName(extName), "TeamID"] = teamId

latestTemplates = teamSeason.dropna(subset=["TeamID"]).copy()
latestTemplates["TeamID"] = latestTemplates["TeamID"].astype(int)
latestTemplates = latestTemplates.sort_values(["TeamID", "Season"]).groupby("TeamID", as_index=False).tail(1)
medianNumeric = latestTemplates.select_dtypes(include=[np.number]).median(numeric_only=True)

profiles2026 = season2026.merge(latestTemplates, on="TeamID", how="left", suffixes=("", "_template"))

for col, value in medianNumeric.items():
    if col in profiles2026.columns:
        profiles2026[col] = profiles2026[col].fillna(value)
    else:
        profiles2026[col] = value

# use the latest historical row as a template, then overwrite it with 2026 KenPom metrics.
profiles2026["Season"] = 2026
profiles2026["Mapped ESPN Team Name"] = profiles2026["TeamName"]
profiles2026["Full Team Name"] = profiles2026["Full Team Name"].fillna(profiles2026["TeamName"])

columnMap = {
    "Adjusted Temo": "AdjTempo",
    "Adjusted Tempo Rank": "RankAdjTempo",
    "Adjusted Offensive Efficiency": "AdjOE",
    "Adjusted Offensive Efficiency Rank": "RankAdjOE",
    "Adjusted Defensive Efficiency": "AdjDE",
    "Adjusted Defensive Efficiency Rank": "RankAdjDE",
    "Raw Tempo": "Tempo",
    "Raw Tempo Rank": "RankTempo",
    "Raw Offensive Efficiency": "OE",
    "Raw Offensive Efficiency Rank": "RankOE",
    "Raw Defensive Efficiency": "DE",
    "Raw Defensive Efficiency Rank": "RankDE",
    "Off2PtFG": "FG2Pct",
    "RankOff2PtFG": "RankFG2Pct",
    "Off3PtFG": "FG3Pct",
    "RankOff3PtFG": "RankFG3Pct",
    "OffFT": "FTPct",
    "RankOffFT": "RankFTPct",
    "Def2PtFG": "OppFG2Pct",
    "RankDef2PtFG": "RankOppFG2Pct",
    "Def3PtFG": "OppFG3Pct",
    "RankDef3PtFG": "RankOppFG3Pct",
    "DefFT": "OppFTPct",
    "RankDefFT": "RankOppFTPct",
}

for newCol, sourceCol in columnMap.items():
    profiles2026[newCol] = profiles2026[sourceCol]

for base in [
    "Tempo", "RankTempo", "AdjTempo", "RankAdjTempo", "OE", "RankOE",
    "AdjOE", "RankAdjOE", "DE", "RankDE", "AdjDE", "RankAdjDE", "AdjEM", "RankAdjEM"
]:
    profiles2026[f"Pre-Tournament.{base}"] = profiles2026[base]

# week 5 built these shooting-rate features from box score data, so approximate them from 2026 split rates.
profiles2026["fgPct"] = (
    profiles2026["FG2Pct"] * (1 - profiles2026["FG3Rate"] / 100.0)
    + profiles2026["FG3Pct"] * (profiles2026["FG3Rate"] / 100.0)
).round(1)
profiles2026["threePtPct"] = profiles2026["FG3Pct"].round(1)
profiles2026["ftPct"] = profiles2026["FTPct"].round(1)
profiles2026["eFgPct"] = profiles2026["eFGPct"].round(1)

# 2026 exports do not include raw assists / turnovers, so estimate AST/TO from historical ARate and TOPct.
winnerAstTo = rawGameStats[["Season", "WTeamID", "WAst", "WTO"]].rename(columns={"WTeamID": "TeamID", "WAst": "AST", "WTO": "TO"})
loserAstTo = rawGameStats[["Season", "LTeamID", "LAst", "LTO"]].rename(columns={"LTeamID": "TeamID", "LAst": "AST", "LTO": "TO"})
astToHist = pd.concat([winnerAstTo, loserAstTo], ignore_index=True)
astToHist = astToHist.groupby(["Season", "TeamID"], as_index=False)[["AST", "TO"]].sum()
astToHist = astToHist[astToHist["TO"] > 0].copy()
astToHist["astTo"] = astToHist["AST"] / astToHist["TO"]

astToTrain = astToHist.merge(teamSeason[["Season", "TeamID", "ARate", "TOPct"]], on=["Season", "TeamID"], how="inner")
astToTrain = astToTrain.dropna()
astToTrain["ratio"] = astToTrain["ARate"] / (astToTrain["TOPct"] * 2.5)

astToModel = LinearRegression()
astToModel.fit(astToTrain[["ratio", "ARate", "TOPct"]], astToTrain["astTo"])

profiles2026["ratio"] = profiles2026["ARate"] / (profiles2026["TOPct"] * 2.5)
astToPred = astToModel.predict(profiles2026[["ratio", "ARate", "TOPct"]])
profiles2026["astTo"] = np.clip(astToPred, 0.3, 4.0).round(1)

profiles2026 = profiles2026[profiles2026["TeamID"].notna()].copy()
profiles2026["TeamID"] = profiles2026["TeamID"].astype(int)
profiles2026 = profiles2026.set_index("TeamID", drop=False)

print(f"2026 team profiles ready: {len(profiles2026)} mapped teams")

In [ ]:
# editable matchup interface
# Assumptions used here:
# 1. Thursday first-round games use DayNum 136 and Friday first-round games use DayNum 137.
# 2. Pre-Tournament.* fields are copied from the current 2026 season values.
# 3. Non-KenPom-only fields missing for newer teams fall back to the latest historical median template.

matchups = pd.DataFrame([
    {"date": "2026-03-19", "dayNum": 136, "team1Seed": 8, "team1": "Ohio State", "team2Seed": 9, "team2": "TCU", "tip": "12:15 p.m.", "network": "CBS"},
    {"date": "2026-03-19", "dayNum": 136, "team1Seed": 4, "team1": "Nebraska", "team2Seed": 13, "team2": "Troy", "tip": "12:40 p.m.", "network": "truTV"},
    {"date": "2026-03-19", "dayNum": 136, "team1Seed": 6, "team1": "Louisville", "team2Seed": 11, "team2": "South Florida", "tip": "1:30 p.m.", "network": "TNT"},
    {"date": "2026-03-19", "dayNum": 136, "team1Seed": 5, "team1": "Wisconsin", "team2Seed": 12, "team2": "High Point", "tip": "1:50 p.m.", "network": "TBS"},
    {"date": "2026-03-19", "dayNum": 136, "team1Seed": 1, "team1": "Duke", "team2Seed": 16, "team2": "Siena", "tip": "2:50 p.m.", "network": "CBS"},
    {"date": "2026-03-19", "dayNum": 136, "team1Seed": 5, "team1": "Vanderbilt", "team2Seed": 12, "team2": "McNeese", "tip": "3:15 p.m.", "network": "truTV"},
    {"date": "2026-03-19", "dayNum": 136, "team1Seed": 3, "team1": "Michigan State", "team2Seed": 14, "team2": "North Dakota State", "tip": "4:05 p.m.", "network": "TNT"},
    {"date": "2026-03-19", "dayNum": 136, "team1Seed": 4, "team1": "Arkansas", "team2Seed": 13, "team2": "Hawai'i", "tip": "4:25 p.m.", "network": "TBS"},
    {"date": "2026-03-19", "dayNum": 136, "team1Seed": 6, "team1": "North Carolina", "team2Seed": 11, "team2": "VCU", "tip": "6:50 p.m.", "network": "TNT"},
    {"date": "2026-03-19", "dayNum": 136, "team1Seed": 1, "team1": "Michigan", "team2Seed": 16, "team2": "Howard", "tip": "7:10 p.m.", "network": "CBS"},
    {"date": "2026-03-19", "dayNum": 136, "team1Seed": 6, "team1": "BYU", "team2Seed": 11, "team2": "Texas", "tip": "7:25 p.m.", "network": "TBS"},
    {"date": "2026-03-19", "dayNum": 136, "team1Seed": 7, "team1": "Saint Mary's", "team2Seed": 10, "team2": "Texas A&M", "tip": "7:35 p.m.", "network": "truTV"},
    {"date": "2026-03-19", "dayNum": 136, "team1Seed": 3, "team1": "Illinois", "team2Seed": 14, "team2": "Penn", "tip": "9:25 p.m.", "network": "TNT"},
    {"date": "2026-03-19", "dayNum": 136, "team1Seed": 8, "team1": "Georgia", "team2Seed": 9, "team2": "Saint Louis", "tip": "9:45 p.m.", "network": "CBS"},
    {"date": "2026-03-19", "dayNum": 136, "team1Seed": 3, "team1": "Gonzaga", "team2Seed": 14, "team2": "Kennesaw State", "tip": "10 p.m.", "network": "TNT"},
    {"date": "2026-03-19", "dayNum": 136, "team1Seed": 2, "team1": "Houston", "team2Seed": 15, "team2": "Idaho", "tip": "10:10 p.m.", "network": "truTV"},
    {"date": "2026-03-20", "dayNum": 137, "team1Seed": 7, "team1": "Kentucky", "team2Seed": 10, "team2": "Santa Clara", "tip": "12:15 p.m.", "network": "CBS"},
    {"date": "2026-03-20", "dayNum": 137, "team1Seed": 5, "team1": "Texas Tech", "team2Seed": 12, "team2": "Akron", "tip": "12:40 p.m.", "network": "truTV"},
    {"date": "2026-03-20", "dayNum": 137, "team1Seed": 1, "team1": "Arizona", "team2Seed": 16, "team2": "Long Island University", "tip": "1:35 p.m.", "network": "TNT"},
    {"date": "2026-03-20", "dayNum": 137, "team1Seed": 3, "team1": "Virginia", "team2Seed": 14, "team2": "Wright State", "tip": "1:50 p.m.", "network": "TBS"},
    {"date": "2026-03-20", "dayNum": 137, "team1Seed": 2, "team1": "Iowa State", "team2Seed": 15, "team2": "Tennessee State", "tip": "2:50 p.m.", "network": "CBS"},
    {"date": "2026-03-20", "dayNum": 137, "team1Seed": 4, "team1": "Alabama", "team2Seed": 13, "team2": "Hofstra", "tip": "3:15 p.m.", "network": "truTV"},
    {"date": "2026-03-20", "dayNum": 137, "team1Seed": 8, "team1": "Villanova", "team2Seed": 9, "team2": "Utah State", "tip": "4:10 p.m.", "network": "TNT"},
    {"date": "2026-03-20", "dayNum": 137, "team1Seed": 6, "team1": "Tennessee", "team2Seed": 11, "team2": "Miami (Ohio)", "tip": "4:25 p.m.", "network": "TBS"},
    {"date": "2026-03-20", "dayNum": 137, "team1Seed": 8, "team1": "Clemson", "team2Seed": 9, "team2": "Iowa", "tip": "6:50 p.m.", "network": "TNT"},
    {"date": "2026-03-20", "dayNum": 137, "team1Seed": 5, "team1": "St. John's", "team2Seed": 12, "team2": "UNI", "tip": "7:10 p.m.", "network": "CBS"},
    {"date": "2026-03-20", "dayNum": 137, "team1Seed": 7, "team1": "UCLA", "team2Seed": 10, "team2": "UCF", "tip": "7:25 p.m.", "network": "TBS"},
    {"date": "2026-03-20", "dayNum": 137, "team1Seed": 2, "team1": "Purdue", "team2Seed": 15, "team2": "Queens", "tip": "7:35 p.m.", "network": "truTV"},
    {"date": "2026-03-20", "dayNum": 137, "team1Seed": 1, "team1": "Florida", "team2Seed": 16, "team2": "Prairie View A&M", "tip": "9:25 p.m.", "network": "TNT"},
    {"date": "2026-03-20", "dayNum": 137, "team1Seed": 4, "team1": "Kansas", "team2Seed": 13, "team2": "Cal Baptist", "tip": "9:45 p.m.", "network": "CBS"},
    {"date": "2026-03-20", "dayNum": 137, "team1Seed": 2, "team1": "UConn", "team2Seed": 15, "team2": "Furman", "tip": "10 p.m.", "network": "TBS"},
    {"date": "2026-03-20", "dayNum": 137, "team1Seed": 7, "team1": "Miami (Fla.)", "team2Seed": 10, "team2": "Missouri", "tip": "10:10 p.m.", "network": "truTV"},
])
matchups["gameOrder"] = np.arange(len(matchups))

display(matchups)

In [ ]:
specialDiffBases = {"Seed", "fgPct", "threePtPct", "ftPct", "eFgPct", "astTo"}
profileDiffBases = [
    feature[:-5]
    for feature in featureCols
    if feature.endswith("_diff") and feature[:-5] not in specialDiffBases
]

featureRows = []
predictionMeta = []

for game in matchups.to_dict(orient="records"):
    team1Id = resolveTeamId(game["team1"])
    team2Id = resolveTeamId(game["team2"])

    if team1Id not in profiles2026.index or team2Id not in profiles2026.index:
        raise KeyError(f"Missing 2026 team profile for matchup: {game['team1']} vs {game['team2']}")

    team1Profile = profiles2026.loc[team1Id]
    team2Profile = profiles2026.loc[team2Id]

    row = {feature: 0.0 for feature in featureCols}
    row["DayNum"] = game["dayNum"]
    row["team1_Seed"] = game["team1Seed"]
    row["team2_Seed"] = game["team2Seed"]
    row["Seed_diff"] = game["team1Seed"] - game["team2Seed"]

    for base in profileDiffBases:
        row[f"{base}_diff"] = float(team1Profile[base]) - float(team2Profile[base])

    row["teamFgPct"] = float(team1Profile["fgPct"])
    row["teamThreePtPct"] = float(team1Profile["threePtPct"])
    row["teamFtPct"] = float(team1Profile["ftPct"])
    row["teamEFgPct"] = float(team1Profile["eFgPct"])
    row["teamAstTo"] = float(team1Profile["astTo"])

    row["oppFgPct"] = float(team2Profile["fgPct"])
    row["oppThreePtPct"] = float(team2Profile["threePtPct"])
    row["oppFtPct"] = float(team2Profile["ftPct"])
    row["oppEFgPct"] = float(team2Profile["eFgPct"])
    row["oppAstTo"] = float(team2Profile["astTo"])

    row["fgPctDiff"] = round(row["teamFgPct"] - row["oppFgPct"], 1)
    row["threePtPctDiff"] = round(row["teamThreePtPct"] - row["oppThreePtPct"], 1)
    row["ftPctDiff"] = round(row["teamFtPct"] - row["oppFtPct"], 1)
    row["eFgPctDiff"] = round(row["teamEFgPct"] - row["oppEFgPct"], 1)
    row["astToDiff"] = round(row["teamAstTo"] - row["oppAstTo"], 1)

    featureRows.append(row)
    predictionMeta.append({
        **game,
        "gameOrder": game["gameOrder"],
        "team1Id": team1Id,
        "team2Id": team2Id,
        "team1External": team1Profile["TeamName"],
        "team2External": team2Profile["TeamName"],
    })

def probabilityToDecimalOdds(probability):
    clipped = np.clip(probability, 1e-6, 1 - 1e-6)
    return 1 / clipped


def probabilityToAmericanOdds(probability):
    clipped = np.clip(probability, 1e-6, 1 - 1e-6)
    odds = np.where(
        clipped >= 0.5,
        -100 * clipped / (1 - clipped),
        100 * (1 - clipped) / clipped,
    )
    return np.rint(odds).astype(int)


def formatAmericanOdds(odds):
    return np.where(odds > 0, "+" + odds.astype(str), odds.astype(str))


predictionX = pd.DataFrame(featureRows).reindex(columns=featureCols)
team1WinProb = bestModel.predict_proba(predictionX)[:, 1]

predictions = pd.DataFrame(predictionMeta)
predictions["team1WinProb"] = team1WinProb
predictions["team2WinProb"] = 1 - predictions["team1WinProb"]
predictions["team1WinPct"] = predictions["team1WinProb"] * 100
predictions["team2WinPct"] = predictions["team2WinProb"] * 100

# These are fair odds implied directly by the model probability, not sportsbook market prices.
predictions["team1FairDecimalOdds"] = probabilityToDecimalOdds(predictions["team1WinProb"])
predictions["team2FairDecimalOdds"] = probabilityToDecimalOdds(predictions["team2WinProb"])
predictions["team1FairAmericanOdds"] = probabilityToAmericanOdds(predictions["team1WinProb"])
predictions["team2FairAmericanOdds"] = probabilityToAmericanOdds(predictions["team2WinProb"])
predictions["team1FairAmericanOddsDisplay"] = formatAmericanOdds(predictions["team1FairAmericanOdds"])
predictions["team2FairAmericanOddsDisplay"] = formatAmericanOdds(predictions["team2FairAmericanOdds"])

# This is the pure higher-probability side.
predictions["winnerByProbability"] = np.where(predictions["team1WinProb"] >= 0.5, predictions["team1"], predictions["team2"])
predictions["winnerProb"] = predictions[["team1WinProb", "team2WinProb"]].max(axis=1)
predictions["winnerProbPct"] = predictions["winnerProb"] * 100
predictions["winnerFairDecimalOdds"] = probabilityToDecimalOdds(predictions["winnerProb"])
predictions["winnerFairAmericanOdds"] = probabilityToAmericanOdds(predictions["winnerProb"])
predictions["winnerFairAmericanOddsDisplay"] = formatAmericanOdds(predictions["winnerFairAmericanOdds"])

# This follows the saved week 8 threshold exactly.
predictions["modelThreshold"] = modelThreshold
predictions["winnerBySavedThreshold"] = np.where(predictions["team1WinProb"] >= modelThreshold, predictions["team1"], predictions["team2"])
predictions["savedThresholdWinnerProb"] = np.where(predictions["team1WinProb"] >= modelThreshold, predictions["team1WinProb"], predictions["team2WinProb"])
predictions["savedThresholdWinnerProbPct"] = predictions["savedThresholdWinnerProb"] * 100
predictions["savedThresholdWinnerFairDecimalOdds"] = probabilityToDecimalOdds(predictions["savedThresholdWinnerProb"])
predictions["savedThresholdWinnerFairAmericanOdds"] = probabilityToAmericanOdds(predictions["savedThresholdWinnerProb"])
predictions["savedThresholdWinnerFairAmericanOddsDisplay"] = formatAmericanOdds(predictions["savedThresholdWinnerFairAmericanOdds"])

predictions = predictions[[
    "gameOrder",
    "date", "tip", "network", "team1Seed", "team1", "team1External", "team1Id",
    "team2Seed", "team2", "team2External", "team2Id",
    "team1WinProb", "team1WinPct", "team1FairDecimalOdds", "team1FairAmericanOdds", "team1FairAmericanOddsDisplay",
    "team2WinProb", "team2WinPct", "team2FairDecimalOdds", "team2FairAmericanOdds", "team2FairAmericanOddsDisplay",
    "winnerByProbability", "winnerProb", "winnerProbPct", "winnerFairDecimalOdds", "winnerFairAmericanOdds", "winnerFairAmericanOddsDisplay",
    "winnerBySavedThreshold", "savedThresholdWinnerProb", "savedThresholdWinnerProbPct",
    "savedThresholdWinnerFairDecimalOdds", "savedThresholdWinnerFairAmericanOdds", "savedThresholdWinnerFairAmericanOddsDisplay",
    "modelThreshold"
]]

predictions = predictions.sort_values(["gameOrder"]).drop(columns=["gameOrder"]).reset_index(drop=True)
display(predictions)

In [ ]:
# save predictions for easy reuse outside the notebook
processedDir = projectDir / "data" / "processed"
processedDir.mkdir(parents=True, exist_ok=True)

predictionsPath = processedDir / "game_predictor_2026_first_round_predictions.csv"
predictions.to_csv(predictionsPath, index=False)

print(f"Saved predictions to: {predictionsPath}")